In [1]:
# =========================================
# IMPORT
# =========================================
import pandas as pd
import numpy as np
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import mean_squared_error, silhouette_score

# =========================================
# LOAD DATA
# =========================================
reservations = pd.read_csv(r"C:\dw_csv\Fact_Reservation.csv")
prices = pd.read_csv(r"C:\dw_csv\Fact_ServicePrice.csv")

df = reservations.merge(
    prices,
    left_on="id_service",
    right_on="id_serviceFK",
    how="left"
)

# =========================================
# DATA PREPARATION
# =========================================
df['date'] = pd.to_datetime(df['id_date'], errors='coerce')
df['revenue'] = df['price']
df = df.dropna()

# =========================================
# 1. DEMAND FORECASTING (Exponential Smoothing weekly)
# =========================================
weekly_demand = df.groupby(pd.Grouper(key='date', freq='W')).size().fillna(0)
print("\n📊 DEMAND FORECASTING")

if len(weekly_demand) >= 4:
    try:
        model_es = ExponentialSmoothing(weekly_demand, trend='add', seasonal=None).fit()
        pred_es = model_es.forecast(4)
        mse_es = mean_squared_error(weekly_demand[-4:], pred_es)
    except:
        mse_es = np.inf
else:
    mse_es = np.inf

print("Exponential Smoothing MSE:", mse_es)
best_demand = "Exponential Smoothing"
print("✅ Best Demand Model:", best_demand)

# =========================================
# 2. CANCELLATION PREDICTION
# =========================================
print("\n🎯 CANCELLATION PREDICTION")
df['cancelled'] = df['id_status'].apply(lambda x: 1 if x == 2 else 0)
class_counts = df['cancelled'].value_counts()
print("Distribution des classes:\n", class_counts)

if len(class_counts) < 2:
    print("⚠️ Impossible de faire de la classification : il n'y a qu'une seule classe.")
else:
    print("✅ Classes équilibrées, on peut entraîner un modèle.")

# =========================================
# 3. REVENUE OPTIMIZATION (Gradient Boosting)
# =========================================
print("\n💰 REVENUE OPTIMIZATION")
demand_price = df.groupby('price').agg({
    'price': 'count',
    'revenue': 'sum'
}).rename(columns={'price':'demand', 'revenue':'total_revenue'}).reset_index()

demand_price['avg_revenue_per_booking'] = demand_price['total_revenue'] / demand_price['demand']

X_reg = demand_price[['price', 'avg_revenue_per_booking']]
y_reg = demand_price['demand']

gbr = GradientBoostingRegressor(random_state=42).fit(X_reg, y_reg)
pred_gbr = gbr.predict(X_reg)
mse_gbr = mean_squared_error(y_reg, pred_gbr)

print("Gradient Boosting MSE:", mse_gbr)
best_reg = "Gradient Boosting"
print("✅ Best Regression Model:", best_reg)

# =========================================
# 4. SEGMENTATION
# =========================================
print("\n👥 SEGMENTATION")
df['nb_res'] = 1
customer = df.groupby('id_service').agg({
    'nb_res': 'sum',
    'price': 'sum'
}).rename(columns={'price': 'spending'})

kmeans = KMeans(n_clusters=3, random_state=42)
labels_k = kmeans.fit_predict(customer)

agg = AgglomerativeClustering(n_clusters=3)
labels_a = agg.fit_predict(customer)

score_k = silhouette_score(customer, labels_k)
score_a = silhouette_score(customer, labels_a)

print("KMeans Score:", score_k)
print("Agglomerative Score:", score_a)
best_cluster = "KMeans" if score_k > score_a else "Agglomerative"
print("✅ Best Clustering Model:", best_cluster)

# =========================================
# 5. FINANCIAL FORECASTING (Exponential Smoothing weekly)
# =========================================
print("\n📈 FINANCIAL FORECASTING")
weekly_revenue = df.groupby(pd.Grouper(key='date', freq='W'))['revenue'].sum().fillna(0)

if len(weekly_revenue) >= 4:
    try:
        model_rev_es = ExponentialSmoothing(weekly_revenue, trend='add', seasonal=None).fit()
        pred_rev_es = model_rev_es.forecast(4)
        mse_rev_es = mean_squared_error(weekly_revenue[-4:], pred_rev_es)
    except:
        mse_rev_es = np.inf
else:
    mse_rev_es = np.inf

print("Exponential Smoothing MSE:", mse_rev_es)
best_fin = "Exponential Smoothing"
print("✅ Best Financial Model:", best_fin)

print("\n🚀 ALL MODELS EXECUTED SUCCESSFULLY")


📊 DEMAND FORECASTING
Exponential Smoothing MSE: inf
✅ Best Demand Model: Exponential Smoothing

🎯 CANCELLATION PREDICTION
Distribution des classes:
 cancelled
0    6795
Name: count, dtype: int64
⚠️ Impossible de faire de la classification : il n'y a qu'une seule classe.

💰 REVENUE OPTIMIZATION
Gradient Boosting MSE: 3.5606241035753095
✅ Best Regression Model: Gradient Boosting

👥 SEGMENTATION
KMeans Score: 0.8893631552092451
Agglomerative Score: 0.9116633505945546
✅ Best Clustering Model: Agglomerative

📈 FINANCIAL FORECASTING
Exponential Smoothing MSE: inf
✅ Best Financial Model: Exponential Smoothing

🚀 ALL MODELS EXECUTED SUCCESSFULLY
